In [ ]:
!pip install pymupdf faiss-cpu wandb datasets -q

In [ ]:
import fitz   # pymupdf
import pandas as pd
import numpy as np
import re
import os

def extract_pdf_chunks(pdf_path, chunk_size=80, overlap=20):
    
    doc   = fitz.open(pdf_path)
    pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        text = re.sub(r'\s+', ' ', text)          
        text = re.sub(r'[^\w\s\.\,\;\:\-\(\)]', ' ', text)  # remove special chars
        text = text.strip()

        if len(text) > 50:   # skip very short pages
            pages.append({
                'page': page_num + 1,
                'text': text
            })

    doc.close()

    full_text = ' '.join([p['text'] for p in pages])
    words = full_text.split()
    
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        chunk_text  = ' '.join(chunk_words)

        if len(chunk_text) > 50:   # skip tiny chunks
            chunks.append(chunk_text)

        i += (chunk_size - overlap)   # overlap for context continuity

    return chunks


PDF_DIR = '/kaggle/input/datasets/aadity7531/rag-dataset'

pdfs = {
    'Physics_11_part1':   'NCERT-Class-11-Physics-Part-1.pdf',
    'Physics_11_part2':   'NCERT-Class-11-Physics-Part-2.pdf',
    'Physics_12_part1':   'NCERT-Class-12-Physics-Part-1.pdf',
    'Physics_12_part2':   'NCERT-Class-12-Physics-Part-2.pdf',
    'Chemistry_11_part1': 'NCERT-Class-11-Chemistry-Part-1.pdf',
    'Chemistry_11_part2': 'NCERT-Class-11-Chemistry-Part-2.pdf',
    'Chemistry_12_part1': 'NCERT-Class-12-Chemistry-Part-1.pdf',
    'Chemistry_12_part2': 'NCERT-Class-12-Chemistry-Part-2.pdf',
    'geography1': 'Fundamental of Physical Geography (Class XI) 2.pdf',
    'geography2': 'India Physical Environment (Class XI) 2.pdf',
    'geography3': '/kaggle/input/datasets/aadity7531/rag-dataset/Practical Work in Geography Part 1.pdf',
    'class9': '/kaggle/input/datasets/aadity7531/rag-dataset/class 9 abc.pdf',
    'history1': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-10-History.pdf',
    'history2': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-11-History.pdf',
    'Biology_11': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-11-Biology.pdf',
    'Biology_12': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-Biology.pdf',
    'history_12_1': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-1 (1).pdf',
    'History_12_2': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-2.pdf',
    'History_12_3': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-3.pdf',
    
}

all_chunks = []

for subject, filename in pdfs.items():
    path = os.path.join(PDF_DIR, filename)
    
    chunks = extract_pdf_chunks(path, chunk_size=100, overlap=30)

    for chunk in chunks:
        all_chunks.append({
            'subject': subject,
            'text':    chunk
        })

    print(f"{subject}: {len(chunks)} chunks extracted")

# Save to dataframe
knowledge_df = pd.DataFrame(all_chunks)
print(knowledge_df.iloc[10]['text'][:300])

In [ ]:
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

docs = knowledge_df['text'].tolist()

# TF-IDF vectorizer on NCERT text
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    strip_accents='unicode'
)

doc_vecs = tfidf.fit_transform(docs).toarray().astype(np.float32)
doc_vecs  = normalize(doc_vecs, norm='l2')

# FAISS index
DIM   = doc_vecs.shape[1]
index = faiss.IndexFlatIP(DIM)
index.add(doc_vecs)

print(f"FAISS ready — {index.ntotal} knowledge chunks, dim={DIM}")

In [ ]:
def get_ncert_context(prompt: str, options: dict, top_k: int = 3) -> str:

    query = prompt + ' ' + ' '.join(options.values())
    query = query[:500]   # limit query length

    vec = tfidf.transform([query]).toarray().astype(np.float32)
    vec = normalize(vec, norm='l2')

    scores, idxs = index.search(vec, top_k + 2)

    parts = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        if score < 0.15:       # skip irrelevant chunks
            continue
        if len(parts) >= top_k:
            break

        chunk = docs[int(idx)][:250]   # truncate for token budget
        parts.append(chunk)

    return ' || '.join(parts) if parts else 'No relevant context.'



test_q = "What is the relationship between Hamiltonians in quantum mechanics?"
test_opts = {'A': 'same energy', 'B': 'higher energy', 'C': 'different spin', 'D': 'different energy', 'E': 'lower energy'}

ctx = get_ncert_context(test_q, test_opts)
print("Query:", test_q)
print("\nRetrieved NCERT context:")
print(ctx[:400])

In [ ]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

OPTIONS  = ['A', 'B', 'C', 'D', 'E']
LABEL2ID = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
ID2LABEL = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

for col in OPTIONS + ['prompt']:
    train[col] = train[col].fillna('none')
    test[col]  = test[col].fillna('none')

train['label'] = train['answer'].map(LABEL2ID)
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

# Build contexts
print("Building NCERT RAG context for train...")
train_ctx = []
for i in range(len(train)):
    opts = {opt: train.iloc[i][opt] for opt in OPTIONS}
    ctx  = get_ncert_context(train.iloc[i]['prompt'], opts)
    train_ctx.append(ctx)
    if i % 400 == 0:
        print(f"  {i}/2000")

train['rag_context'] = train_ctx

test_ctx = []
for i in range(len(test)):
    opts = {opt: test.iloc[i][opt] for opt in OPTIONS}
    ctx  = get_ncert_context(test.iloc[i]['prompt'], opts)
    test_ctx.append(ctx)
    if i % 100 == 0:
        print(f"  {i}/500")

test['rag_context'] = test_ctx

real_ctx = (train['rag_context'] != 'No relevant context.').sum()
print(f"Questions with real NCERT context: {real_ctx}/2000 ({real_ctx/20:.1f}%)")

In [ ]:
import torch
from transformers import AutoTokenizer
from datasets import Dataset

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = '/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1'
MAX_LEN    = 256
EPOCHS     = 2
LR         = 1e-5
BATCH_SIZE = 4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    all_input_ids      = []
    all_attention_mask = []

    for i in range(len(examples['prompt'])):
        
        ctx    = str(examples['rag_context'][i])[:150]
        prompt = str(examples['prompt'][i])

        # Format: "Context: <ncert_text> Question: <prompt>"
        first_seq = f"Context: {ctx} Question: {prompt}"

        opt_ids = []
        opt_att = []

        for opt in OPTIONS:
            second_seq = str(examples[opt][i])

            enc = tokenizer(
                first_seq,
                second_seq,
                max_length=MAX_LEN,
                truncation=True,
                padding='max_length',
                return_tensors=None,
            )
            opt_ids.append([int(x) for x in enc['input_ids']])
            opt_att.append([int(x) for x in enc['attention_mask']])

        all_input_ids.append(opt_ids)
        all_attention_mask.append(opt_att)

    return {
        'input_ids':      all_input_ids,
        'attention_mask': all_attention_mask,
    }

In [ ]:
from sklearn.model_selection import train_test_split

KEEP     = OPTIONS + ['prompt', 'rag_context']
KEEP_LBL = KEEP + ['label']

# Use full 2000 for training
train_hf = Dataset.from_pandas(train[KEEP_LBL].copy())
test_hf  = Dataset.from_pandas(test[KEEP].copy())


train_tok = train_hf.map(tokenize_fn, batched=True, batch_size=32, remove_columns=KEEP)

test_tok  = test_hf.map(tokenize_fn, batched=True, batch_size=32, remove_columns=KEEP)

FEAT = ['input_ids', 'attention_mask']
train_tok.set_format(type='torch', columns=FEAT + ['label'])
test_tok.set_format( type='torch', columns=FEAT)

print("Done!")
print("Shape:", train_tok[0]['input_ids'].shape)  # (5, 256)

In [ ]:
import gc
import wandb
import torch
import torch.nn as nn
import pandas as pd
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForMultipleChoice, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

def collate_train(features):
    return {
        'input_ids':      torch.stack([f['input_ids']      for f in features]).long(),
        'attention_mask': torch.stack([f['attention_mask'] for f in features]).long(),
        'labels':         torch.stack([f['label']          for f in features]).long(),
    }

def collate_test(features):
    return {
        'input_ids':      torch.stack([f['input_ids']      for f in features]).long(),
        'attention_mask': torch.stack([f['attention_mask'] for f in features]).long(),
    }


train_loader = DataLoader(
    train_tok, batch_size=BATCH_SIZE,
    shuffle=True,  collate_fn=collate_train
)
test_loader = DataLoader(
    test_tok, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_test,
)


wandb.login(key="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")
wandb.init(
    project='24f1002052-t22026',
    name='deberta-NCERT-RAG',
    config={
        'model':     MODEL_NAME,
        'knowledge': 'NCERT Physics+Chemistry 11+12',
        'epochs':    EPOCHS,
        'lr':        LR,
        'max_len':   MAX_LEN
    }
)


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

model     = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME, ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS

scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

best_loss  = float('inf')
best_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss  = 0.0
    valid_steps = 0

    for step, batch in enumerate(train_loader):
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop('labels')
        optimizer.zero_grad()
        
        out  = model(**batch)
        loss = criterion(out.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()

        total_loss  += loss.item()
        valid_steps += 1

        if step % 50 == 0:
            print(f"  Step {step}/{len(train_loader)} loss={loss.item():.4f}")

    avg = total_loss / max(valid_steps, 1)
    wandb.log({'epoch': epoch+1, 'train_loss': avg})

    if avg < best_loss:
        best_loss  = avg
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
       
wandb.finish()
print("Training done!")


model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out   = model(**batch)
        preds = torch.argmax(out.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

pred_labels = [ID2LABEL[int(p)] for p in all_preds]

sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sub['Prediction'] = pred_labels
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.head())